In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / "python"))

Project directory set to: /home/vertyx/Documents/hochschule/dermascan


In [2]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
import cv2

from dataset_manager import DatasetManager
from pytorch_model_exporter import PytorchModelExporter
from preprocessing import image_filters_opencv
from pipeline.dataclasses.enums import ModelType

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 

In [4]:
dataset_manager = DatasetManager(data_dir=str(proj_dir / "data"))
dataset_manager.load_cleaned_metadata()
image_id = "ISIC_0033238"
img_path = proj_dir / "data" / dataset_manager.df.loc[dataset_manager.df["image_id"] == image_id].iloc[0]["image_path"]
img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)

## Exporting the trained Models for Lesion Classification

In [5]:
base_results = proj_dir / "results" / "output_node_grid_resnet18_efficientnet_v2_s_2026-05-18_21-43"

configs = [
    {
        "name": "rgb_efficientnet_v2_s_b_2node",
        "arch": ModelType.EFFICIENTNET_V2_S,
        "input_channels": 3,
        "output_channels": 2,
        "model_path": base_results / "rgb_efficientnet_v2_s_b_2node_rgb" / "model.pt",
        "output_path": proj_dir / "flutter" / "assets" / "models" / "rgb_efficientnet_v2_s_b_2node.onnx",
        "sample_img": img,
    },
    {
        "name": "rgb_efficientnet_v2_s_a_1node",
        "arch": ModelType.EFFICIENTNET_V2_S,
        "input_channels": 3,
        "output_channels": 1,
        "model_path": base_results / "rgb_efficientnet_v2_s_a_1node_rgb" / "model.pt",
        "output_path": proj_dir / "flutter" / "assets" / "models" / "rgb_efficientnet_v2_s_a_1node.onnx",
        "sample_img": img,
    },
]


In [6]:
for cfg in configs:
    print(f"\n=== Exporting {cfg['name']} ===")
    exporter = PytorchModelExporter(input_channels=cfg["input_channels"], output_channels=cfg["output_channels"])    
    img_tensor = exporter.load_sample_tensor(sample_img=cfg["sample_img"], target_shape=(224,224), normalize=True)
    print("Input tensor shape:", img_tensor.shape)

    model_path = cfg["model_path"]
    if not model_path.exists():
        print(f"Warning: pretrained weights not found at {model_path}. Exporting uninitialized model.")
        model_path = None

    torch_model = exporter.build_model_from_arch(arch=cfg["arch"], input_channels=cfg["input_channels"], num_classes=cfg["output_channels"], pretrained_weights=model_path, device=str(DEVICE))

    # create parent dir
    cfg["output_path"].parent.mkdir(parents=True, exist_ok=True)

    exporter.export_torch_model(example_input=img_tensor, save_to=cfg["output_path"])    
    print(f"Exported ONNX to: {cfg['output_path']}")

    dist, to, oo = exporter.compare_model_outputs(img_tensor, cfg["output_path"])    
    print(f"Distance between PyTorch and ONNX outputs: {dist:.6f}%")
    # print predictions
    import numpy as np
    def get_pred(logits):
        exp = np.exp(logits - np.max(logits))
        probs = exp / np.sum(exp)
        idx = np.argmax(probs)
        return idx, probs[idx]

    idx, conf = get_pred(oo[0])
    print(f"ONNX predicted class index: {idx}, confidence: {conf:.3f}")


=== Exporting rgb_efficientnet_v2_s_b_2node ===
Input tensor shape: torch.Size([1, 3, 224, 224])
Exported ONNX to: /home/vertyx/Documents/hochschule/dermascan/flutter/assets/models/rgb_efficientnet_v2_s_b_2node.onnx
Distance between PyTorch and ONNX outputs: 0.000187%
ONNX predicted class index: 0, confidence: 0.831

=== Exporting rgb_efficientnet_v2_s_a_1node ===
Input tensor shape: torch.Size([1, 3, 224, 224])
Exported ONNX to: /home/vertyx/Documents/hochschule/dermascan/flutter/assets/models/rgb_efficientnet_v2_s_a_1node.onnx
Distance between PyTorch and ONNX outputs: 0.000026%
ONNX predicted class index: 0, confidence: 1.000
